**Installing Dependencies**

In [4]:
!pip install -q langchain-huggingface sentence-transformers faiss-cpu langchain-community langchain-groq langchain-text-splitters beautifulsoup4 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00


**Setting up API Key**

In [2]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('Groq API Key:').strip()
os.environ.setdefault("USER_AGENT", "hr-policy-assistant")


Groq API Key:··········


'hr-policy-assistant'

In [5]:
import os, re, time

import pandas as pd
from datetime import datetime

from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [6]:
sources = [
        {
        "title": "GitLab - Time Off Types",
        "url": "https://handbook.gitlab.com/handbook/people-group/time-off-and-absence/time-off-types/",
        "country": "Global",
        "policy_type": "time_off",
    },
    {
        "title": "GitLab - Leave Types",
        "url": "https://handbook.gitlab.com/handbook/people-group/time-off-and-absence/leave-types/",
        "country": "Global",
        "policy_type": "leave",
    },
    {
        "title": "GitLab - US Leave of Absence",
        "url": "https://handbook.gitlab.com/handbook/people-policies/leave-of-absence/us",
        "country": "US",
        "policy_type": "leave",
    },
    {
        "title": "GitLab - Benefits",
        "url": "https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/",
        "country": "Global",
        "policy_type": "benefits",
    },
    {
        "title": "GitLab - France Benefits",
        "url": "https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/france-sas",
        "country": "France",
        "policy_type": "benefits",
    },
    {
        "title": "Amazon - Code of Conduct",
        "url": "https://ir.aboutamazon.com/corporate-governance/documents-charters/code-business-conduct-and-ethics",
        "country": "Global",
        "policy_type": "conduct",
    },
]

**Loading & Cleaning Texts**

In [7]:
def clean_text(text):
    text = re.sub(r"[ \\t]+", " ", text)
    text = re.sub(r"\\n\\s*\\n+", "\\n\\n", text)
    return text.strip()


documents = []

for source in sources:
    try:
        pages = WebBaseLoader(source["url"]).load()
        text = clean_text("\\n\\n".join(page.page_content for page in pages))

        if text:
            documents.append({
                "text": text,
                "metadata": {
                    "title": source["title"],
                    "url": source["url"],
                    "country": source["country"],
                    "policy_type": source["policy_type"],
                },
            })
            print("Loaded:", source["title"], "-", len(text), "characters")
        else:
            print("No text found:", source["title"])

    except Exception as error:
        print("Could not load:", source["title"], "|", error)

print("\\nTotal webpages loaded:", len(documents))

Loaded: GitLab - Time Off Types - 143038 characters
Loaded: GitLab - Leave Types - 132131 characters
Loaded: GitLab - US Leave of Absence - 151806 characters
Loaded: GitLab - Benefits - 137600 characters
Loaded: GitLab - France Benefits - 127619 characters
Loaded: Amazon - Code of Conduct - 54 characters
\nTotal webpages loaded: 6


**Splitting Webpages into Chunks**

In [14]:
from langchain_core.documents import Document

langchain_document = [
    Document(page_content=item['text'], metadata=item['metadata'])
    for item in documents
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 120
)

chunks = splitter.split_documents(langchain_document)

print("Chunks:", len(chunks))
print(pd.Series([c.metadata["title"] for c in chunks]).value_counts())

Chunks: 1054
GitLab - US Leave of Absence    237
GitLab - Time Off Types         217
GitLab - Benefits               210
GitLab - Leave Types            200
GitLab - France Benefits        189
Amazon - Code of Conduct          1
Name: count, dtype: int64


**Embeddings & Vector Store**

In [15]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {'device': 'cpu'}
)

vectorstore = FAISS.from_documents(chunks, embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
# Setting up Retriever
retriever = vectorstore.as_retriever(
    search_kwargs = {'k':4}
)

**Setting Up PromptTemplate**

In [18]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_template(
    """You are an HR policy assistant.

Answer using only the policy context below.
Keep the answer clear and short.
Mention the policy source when possible.
Do not invent facts.

If the answer is not in the context, reply exactly:
"I couldn't find that in the policy documents. Please contact HR."

Policy context:
{context}

Employee question:
{question}

Answer:"""
)

answer_chain = prompt | llm | StrOutputParser()

In [29]:
def ask_question(question):
    # 1. Find relevant policy chunks
    relevant_docs = retriever.invoke(question)

    # 2. Combine their text into one context
    context = "\\n\\n".join(
        doc.page_content for doc in relevant_docs
    )

    # 3. Ask the LLM to answer from that context
    answer = answer_chain.invoke({
        "context": context,
        "question": question,
    })

    print("ANSWER")
    print(answer)

    print("\\nSOURCES")
    for doc in relevant_docs:
        print("-", doc.metadata.get("title", "Unknown source"))
        print(" ", doc.metadata.get("url", ""))